# Cognivision Voice Intelligence
## Notebook 04: CPU-Optimized PyTorch 2D-CNN Model Training
This notebook demonstrates the end-to-end training of our custom PyTorch 2D-CNN on Log-Mel spectrograms with Early Stopping and speaker-independent validation.

In [ ]:
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

root_dir = Path("..").resolve()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from backend.app.models.cnn_emotion_model import EmotionCNN
from backend.app.services.training_service import SpectrogramDataset, training_service
from backend.app.services.dataset_service import dataset_service
from backend.app.utils.constants import EMOTIONS

### 1. Verify Speaker-Independent Dataset Splits

In [ ]:
splits = dataset_service.create_speaker_independent_splits()
print(f"Train samples: {splits['train_samples']}")
print(f"Validation samples: {splits['val_samples']}")
print(f"Test samples: {splits['test_samples']}")
print(f"Speaker Leakage: {splits['speaker_leakage']}")

### 2. Inspect 2D-CNN Architecture

In [ ]:
model = EmotionCNN(num_classes=len(EMOTIONS))
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model Architecture: {model.__class__.__name__}")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(model)

### 3. Checkpoint & Training History Verification

In [ ]:
history = training_service.get_history()
if history.get('epochs'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    ax1.plot(history['epochs'], history['train_loss'], label='Train Loss', color='#06b6d4')
    ax1.plot(history['epochs'], history['val_loss'], label='Validation Loss', color='#f43f5e')
    ax1.set_title("Training vs Validation Loss", fontweight='bold')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(alpha=0.3)

    ax2.plot(history['epochs'], history['train_acc'], label='Train Acc', color='#10b981')
    ax2.plot(history['epochs'], history['val_acc'], label='Val Acc', color='#8b5cf6')
    ax2.set_title("Training vs Validation Accuracy", fontweight='bold')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.legend()
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("Model not trained yet. Run training_service.start_training() or train via UI.")